## **Binary Search Trees**

<font size = "4">

- Recall that we used **hash tables** to store items as key/value pairs. (Similar to Python dictionaries).

- A hash table is a concrete implementation of the **Map** abstract data type.

- Other common names for the Map ADT are: *associative array*, *key-value store*, *symbol table*, or *dictionary*.

- We could in principle implement a Map just using a list or dynamic array. To search for a key, we would need to use a sequential search algorithm (if the keys are unsorted) or an algorithm like binary search (if it is kept sorted).

- We can use the idea behind binary search to implement a Map as a **Binary Search Tree**

### **Structure of Binary Search Tree (BST)**

<font size = "4">

- Each node of the tree will have a *key* and a *value*. The keys are **unique**

- For a given tree, all nodes in the left subtree have keys less than the root. All nodes in the right subtree have keys larger than the root.

- We start with an empty BST. The first added node is the root, then additional nodes are added in the "downward" direction.

- If we create an empty BST and then add the keys 70, 31, 93, 94, 14, 23, 73 (in that order) then we get the following tree

<div style="text-align: center;">
  <img src="files/simpleBST.png" alt="Centered image" width = "250">
  <br>
  <br>
  <figcaption>Keys of each node in the BST. (Values not shown)<br>
  <font size = "1"> Miller, Randum, Yasinovskyy (Problem Solving with Algorithms and Data Structures using Python)</figcaption>
</div>

<br>

<font size = "4">

- We could also have string-valued keys that are ordered lexicographically (alphabetically)

### **BST methods**

<font size = "4">

- We will use a Python dictionary as our model for what methods should be added to the tree.

- We will need to implement methods called `put`, `get`, and `delete`. We will also find it convenient to know how many items are in the tree.

In [ ]:
my_dict = {"a": "apple", "c": "cat", "f": "falafel", "g":"gorilla", "z":"zebra"}

# this is the "put" method
my_dict["t"] = "tarantula" # new key
my_dict["c"] = "cougar" # replace value of existing key

# this is the "get" method
my_var = my_dict["f"]
print(my_var)

# the "delete" method
del my_dict["a"]

# how many key/value pairs are in the dictionary
print(len(my_dict))

print(my_dict)

<font size = "4">

We also want to:

- Test whether a given key belongs to the tree

- We also want to loop over the keys in the tree

In [ ]:
print('t' in my_dict)
print('x' in my_dict)
print()
for k in my_dict:
    print(k)

### **Implementation**

<font size = "4">

The implementation will be quite involved.

- We will implement the `BinarySearchTree` as a linked collection of `TreeNode` objects.

- Because we want to be able to remove keys while maintaining the ordered structure, it will actually be necessary to add edges/pointers **from child to parent**.

- These additional pointers will also complicate the process when adding a node.

- We will implement "dunder" methods to allow us to (i) put/get items using the bracket `[]` notation, (ii) use the boolean `in` operator to test if a key belongs to the tree, (iii) loop over the keys of the tree in a `for` loop, and (iv) determine the size of the tree using the `len` function

In [ ]:
class BinarySearchTree:
    def __init__(self):
        self.root = None
        self.size = 0

    def __len__(self):
        return self.size

    def __iter__(self):
        return self.root.__iter__()

    def __setitem__(self, key, value):
        self.put(key, value)

    def __getitem__(self, key):
        return self.get(key)

    def __contains__(self, key):
        return bool(self._get(key, self.root))
    
    def __delitem__(self,key):
        self.delete(key)


    def put(self, key, value):
        if self.root:
            # tree is non-empty
            # call private _put method
            self._put(key, value, self.root)
        else:
            # tree is empty, create a new TreeNode
            self.root = TreeNode(key, value)
            self.size = self.size + 1


    def _put(self, key, value, current_node):
        if key < current_node.key:
            # goes in left subtree
            if current_node.left_child:
                self._put(key, value, current_node.left_child)
            else:
                # left subtree is empty
                current_node.left_child = TreeNode(key, value, parent=current_node)
                self.size = self.size + 1
        elif key == current_node.key:
            # overwrite value
            current_node.replace_value(key, value, current_node.left_child, current_node.right_child)
        else:
            # goes in right subtree
            if current_node.right_child:
                self._put(key, value, current_node.right_child)
            else:
                current_node.right_child = TreeNode(key, value, parent=current_node)
                self.size = self.size + 1

    def get(self, key):
        result = self._get(key, self.root)
        if result:
            return result.value
        return None
    
    def _get(self, key, current_node):
        if not current_node:
            return None 
        if current_node.key == key:
            return current_node 
        elif key < current_node.key:
            # search left subtree
            return self._get(key, current_node.left_child)
        else:
            return self._get(key, current_node.right_child)

    

    def delete(self, key):
        if self.size > 1:
            node_to_remove = self._get(key, self.root)
            if node_to_remove:
                self._delete(node_to_remove)
                self.size -= 1
            else:
                raise KeyError("Error, key not in tree")
        elif self.size == 1 and self.root.key == key:
            self.root = None 
            self.size -= 1
        else:
            raise KeyError("Error, key not in tree")

    def _delete(self, current_node):
        if current_node.is_leaf(): # no children
            self._delete_leaf(current_node)
        elif current_node.has_children(): # two children
            self._delete_two_children(current_node)
        else:
            self._delete_one_child(current_node)
    
    def _delete_leaf(self, current_node):
        if current_node == current_node.parent.left_child:
            current_node.parent.left_child = None 
        else:
            current_node.parent.right_child = None

    def _delete_two_children(self, current_node):
        successor = current_node.find_successor()
        successor.splice_out()
        current_node.key = successor.key 
        current_node.value = successor.value

    def _delete_one_child(self, current_node):
        # case 1: has a left child
        if current_node.left_child:
            if current_node.is_left_child():
                current_node.left_child.parent = current_node.parent
                current_node.parent.left_child = current_node.left_child
            elif current_node.is_right_child():
                current_node.left_child.parent = current_node.parent 
                current_node.parent.right_child = current_node.left_child 
            else:
                # Root node with one left child
                current_node.replace_value(
                    current_node.left_child.key,
                    current_node.left_child.value,
                    current_node.left_child.left_child,
                    current_node.left_child.right_child,
                )
        # case 2: has a right child
        else:
            if current_node.is_left_child():
                current_node.right_child.parent = current_node.parent
                current_node.parent.left_child = current_node.right_child
            elif current_node.is_right_child():
                current_node.right_child.parent = current_node.parent
                current_node.parent.right_child = current_node.right_child
            else:
                current_node.replace_value(
                    current_node.right_child.key,
                    current_node.right_child.value,
                    current_node.right_child.left_child,
                    current_node.right_child.right_child,
                )


class TreeNode:
    def __init__(self, key, value, left=None, right=None, parent=None):
        self.key = key
        self.value = value
        self.left_child = left
        self.right_child = right
        self.parent = parent

    def __repr__(self):
        return f"TreeNode with key={self.key}, value={self.value}"

    def __iter__(self):
        # in-order traversal
        if self:
            if self.left_child:
                for elem in self.left_child:
                    yield elem 
            yield self.key 
            if self.right_child:
                for elem in self.right_child:
                    yield elem

    def is_left_child(self):
        return self.parent and self.parent.left_child is self

    def is_right_child(self):
        return self.parent and self.parent.right_child is self

    def is_root(self):
        return not self.parent

    def is_leaf(self):
        return not (self.right_child or self.left_child)

    def has_any_child(self):
        return self.right_child or self.left_child

    def has_children(self):
        return self.right_child and self.left_child

    def replace_value(self, key, value, left, right):
        self.key = key
        self.value = value
        self.left_child = left
        self.right_child = right
        if self.left_child:
            self.left_child.parent = self
        if self.right_child:
            self.right_child.parent = self

    def find_successor(self):
        successor = None 
        if self.right_child:
            successor = self.right_child.find_min()
        else:
            if self.parent:
                if self.is_left_child():
                    successor = self.parent
                else:
                    self.parent.right_child = None 
                    successor = self.parent.find_successor()
                    self.parent.right_child = self
        return successor
    
    def find_min(self):
        current = self
        while current.left_child:
            current = current.left_child 
        return current 
    
    def splice_out(self):
        if self.is_leaf():
            if self.is_left_child():
                self.parent.left_child = None 
            else:
                self.parent.right_child = None
        elif self.has_any_child():
            if self.left_child:
                if self.is_left_child():
                    self.parent.left_child = self.left_child
                else:
                    self.parent.right_child = self.left_child 
                self.left_child.parent = self.parent
            # fixed typo
            else: 
                if self.is_left_child():
                    self.parent.left_child = self.right_child 
                else:
                    self.parent.right_child = self.right_child 
                self.right_child.parent = self.parent


    


### **Adding/replacing keys (put)**

<font size = "4">

- Adding keys is straightforward. Starting at the tree root, we go to the left or right subtree based on whether the key is smaller or larger. We then repeat the process with the node of the appropriate subtree.

- If we encounter a node with the same key, we overwrite the value.

- Otherwise, once we encounter a node with an empty spot for a child, we add a new node.

<div style="text-align: center;">
  <img src="files/bstput.png" alt="Centered image" width = "350">
  <br>
  <br>
  <figcaption>Inserting a Node with Key = 19<br>
  <font size = "1"> Miller, Randum, Yasinovskyy (Problem Solving with Algorithms and Data Structures using Python)</figcaption>
</div>


### **Retrieving keys (get)**

<font size = "4">

- Also pretty simple. Start at the root, and move left or right depending on whether the key we're seeking is smaller or larger than the root's key.

- If we encounter the key we're looking for, return the node's value.

- If we don't find the key, return `None` (this is different behavior than a Python dictionary, which raises a KeyError)

### **Deleting nodes (delete)**

<font size = "4">

- This is the hard part.

- We need to break it up into 3 cases

    - Deleting a node with no children (easiest)

    - Deleting a node with 1 child (a little harder, a pain to code up)

    - Deleting a node with 2 children (hardest). We will add methods to `TreeNode` to help us: `find_successor`, `find_min`, and `splice_out`.

- Here are visual depictions of all 3 cases:


<div style="text-align: center;">
  <img src="files/bstdel1.png" alt="Centered image" width = "550">
  <br>
  <br>
  <figcaption>Deleting Node 16, a Node without Children<br>
  <font size = "1"> Miller, Randum, Yasinovskyy (Problem Solving with Algorithms and Data Structures using Python)</figcaption>
</div>

<br>
<font size = "4">

<div style="text-align: center;">
  <img src="files/bstdel2.png" alt="Centered image" width = "550">
  <br>
  <br>
  <figcaption>Deleting Node 25, a Node That Has a Single Child<br>
  <font size = "1"> Miller, Randum, Yasinovskyy (Problem Solving with Algorithms and Data Structures using Python)</figcaption>
</div>

<br>

<font size = "4">

<div style="text-align: center;">
  <img src="files/bstdel3.png" alt="Centered image" width = "550">
  <br>
  <br>
  <figcaption>Deleting Node 5, a Node with Two Children<br>
  <font size = "1"> Miller, Randum, Yasinovskyy (Problem Solving with Algorithms and Data Structures using Python)</figcaption>
</div>

<br>

In [ ]:
my_tree = BinarySearchTree()
my_tree["a"] = "a"
my_tree["q"] = "quick"
my_tree["b"] = "brown"
my_tree["f"] = "fox"
my_tree["j"] = "jumps"
my_tree["o"] = "over"
my_tree["t"] = "the"
my_tree["l"] = "lazy"
my_tree["d"] = "dog"

print(my_tree["q"])
print(my_tree["l"])
print("There are {} items in this tree".format(len(my_tree)))
# my_tree.delete("a")
del my_tree["a"]
print("There are {} items in this tree".format(len(my_tree)))

for node in my_tree:
    print(my_tree[node], end=" ")
print()


### **Quick Overview of Computational Cost**

<font size = "4">

- The cost of the methods `put`, `get`, and `delete` is dependent on the height of the tree (the number of levels)

- A **balanced** tree has the same number of nodes in the left and right subtrees. There are around $\log_2(n)$ levels in the tree.

- So for a balanced tree, the worst-case performance of these methods is $\mathcal{O}(\log n)$.

- However, a severely **skewed** tree can occur:

<div style="text-align: center;">
  <img src="files/skewedTree.png" alt="Centered image" width = "300">
  <br>
  <br>
  <figcaption>A skewed binary tree, resulting in poor performance<br>
  <font size = "1"> Miller, Randum, Yasinovskyy (Problem Solving with Algorithms and Data Structures using Python)</figcaption>
</div>

<br>

<font size = "4">

- In this case, the cost of each method becomes $\mathcal{O}(n)$, because we are essentially working with an ordered linked list.

- An **AVL tree** (named after G.M. Adelson-Velskii and E.M. Landis) is a binary search tree that automatically ensures the tree remains balanced, improving the performance.

- Here is a summary of the **worst case** computational costs for different implementations of a Map/Dictionary:

<br>

<div align="center">


| operation | Sorted List (w/ binary search) | Binary Search Tree | AVL Tree |
|:--------:|:--------:|:--------:|:--------:|
| `put`    | $\mathcal{O}(n)$   | $\mathcal{O}(n)$   | $\mathcal{O}(\log n)$    |
| `get`      | $\mathcal{O}(\log n)$      | $\mathcal{O}(n)$     | $\mathcal{O}(\log n)$     |
| `in`      | $\mathcal{O}(\log n)$      | $\mathcal{O}(n)$     | $\mathcal{O}(\log n)$   |
| `del`    | $\mathcal{O}(n)$    | $\mathcal{O}(n)$    | $\mathcal{O}(\log n)$     |

</div>

- A hash table has worst-case performance of $\mathcal{O}(n)$ for all of these methods, but you have to work pretty hard to choose a poor hash function. Usually it's $\mathcal{O}(1)$



In [ ]:
my_tree["p"]